# Deep Learning for Natural Language Processing

# Homework 3 - Spring 2025

# Bert Model

## Angelos Dorotheos Chatzopoulos
## 7115112400027

### Python package PIP installs

In [ ]:
%pip install contractions

In [ ]:
!wget https://figshare.com/ndownloader/files/10798046 -O GoogleNews-vectors-negative300.bin

### Python Packages Imports

In [ ]:
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from IPython.display import display
from wordcloud import WordCloud
import re

import nltk
from nltk.tokenize import TweetTokenizer
from nltk.corpus import stopwords
from nltk import ngrams
import contractions
import string

import warnings
import random
import sys

if not sys.warnoptions:
    warnings.simplefilter("ignore")
    os.environ["PYTHONWARNINGS"] = "ignore::UserWarning"

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import roc_curve, auc, RocCurveDisplay
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

import torch
from torch.utils.data import DataLoader, TensorDataset, RandomSampler
from torch.optim import Adam, AdamW, NAdam, RAdam, RMSprop

from transformers import AutoTokenizer, AutoModelForSequenceClassification, set_seed, get_linear_schedule_with_warmup

from gensim.models import KeyedVectors

import optuna

### Setting a specific seed for `numpy`, `os`, `random` and `pytorch` packages for reproducibility

In [ ]:
seed = 42

os.environ['PYTHONHASHSEED'] = str(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
random.seed(seed)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True)

set_seed(seed)

In [ ]:
torch.cuda.is_available()

## Exploratory Data Analysis (EDA)

### Function that plots a pie chart of the percentage of positive and negative labels of datasets

In [ ]:
def plot_pie_chart(counts, title):
    counts = pd.Series(counts)
    counts.plot(kind='pie', autopct='%1.2f%%', figsize=(8, 8), colors=['darkorange', 'royalblue'])
    plt.title(title)
    plt.ylabel('Labels Distribution')
    plt.show()

### Function that plots a histogram that shows the distribution of tweets length across the dataset

In [ ]:
def plot_histogram(data, x_label, y_label, title):
    plt.figure(figsize=(8, 6))
    plt.hist(data, bins=list(np.arange(0, 40, 5)), edgecolor='black', color='royalblue')
    plt.xlabel(x_label)
    plt.ylabel(y_label)
    plt.title(title)
    plt.xlim(0, 50)
    plt.axvline(data.mean(), color='k', linestyle='dashed', linewidth=1)
    _, max_ylim = plt.ylim()
    plt.text(data.mean()*1.1, max_ylim*0.9, 'Mean: {:.2f}'.format(data.mean()))
    plt.tight_layout()
    plt.show()

### Function that plots a wordcloud with the most common words of the dataset

In [ ]:
def create_wordcloud(text):
    all_text = " ".join(text)
    wordcloud = WordCloud(width=800, height=400, background_color='white', random_state=42).generate(all_text)
    plt.figure(figsize=(12, 10))
    plt.imshow(wordcloud, interpolation="bilinear")
    plt.axis("off")
    plt.show()

### Function that plots a bar graph. Used for min-max tweets length and for finding the most common unigrams of the dataset

In [ ]:
def plot_bar(categories, values, x_label, y_label, title, rotate=False, log_scale=False, show_vals=False, bary=False):
    plt.figure(figsize=(8, 6))
    
    if bary:
        bars = plt.barh(categories, values, edgecolor='black', color='royalblue')
    else:
        bars = plt.bar(categories, values, edgecolor='black', color='royalblue')
        
    plt.title(title)
    plt.xlabel(x_label)
    plt.ylabel(y_label)
    
    if show_vals:
         for bar in bars:
            plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + bar.get_height() * 0.01, bar.get_height(),
                ha='center', va='bottom', color='black', fontsize=10)

    if log_scale:
        plt.yscale('log')

    if rotate:
        plt.xticks(rotation=-60)
        
    plt.tight_layout()
    plt.show()

### Function that plots two bar graphs. It is used to count the tweets bigrams and trigrams on the dataset

In [ ]:
def subplot_bars(plot_1_values, plot_2_values, title, rotate=False, bary=False):
    _, axis = plt.subplots(1, 2, figsize=(18, 8))
    
    plt.suptitle(title, fontsize=15)
    
    if bary:
        axis[0].barh(plot_1_values[0], plot_1_values[1], edgecolor='black', color='royalblue')
    else:
        axis[0].bar(plot_1_values[0], plot_1_values[1], edgecolor='black', color='royalblue')

    axis[0].set_xlabel(plot_1_values[2])
    axis[0].set_ylabel(plot_1_values[3])
    
    if rotate:
        axis[0].tick_params(axis='x', labelrotation =-60)
    
    if bary:
        axis[1].barh(plot_2_values[0], plot_2_values[1], edgecolor='black', color='royalblue')
    else:
        axis[1].bar(plot_2_values[0], plot_2_values[1], edgecolor='black', color='royalblue')

    axis[1].set_xlabel(plot_2_values[2])
    axis[1].set_ylabel(plot_2_values[3])

    if rotate:
        axis[1].tick_params(axis='x', labelrotation =-60)
    
    plt.tight_layout()
    plt.show()

### Function that calculates for each row the length of the tweet text

In [ ]:
def get_tweets_length(df):
    return df['Text'].str.split().str.len()

### Function that calculates and returns the top n-grams of a dataframe

In [ ]:
def get_top_tokens(df, top, n=1):
    two_three_dots = ['..', '...']
    count_phrases = [word for tweet in df['Text'] for word in tweet.split() if word not in string.punctuation and word not in two_three_dots]
    ngram_2_tweets = pd.Series([word for word in ngrams(count_phrases, n)]).value_counts()
    top_10_phrases = ngram_2_tweets[:top]
    return [' '.join(x) for x in top_10_phrases.index], list(top_10_phrases)

### Function that finds the minimum and maximum tweet length and returns a new dataframe

In [ ]:
def get_min_max_tweets(t_length):
    return pd.DataFrame({
        'Minimum Tweet Length': [t_length.min()],
        'Maximum Tweet Length': [t_length.max()],
    })

### Load train, validation and test datasets from CSV files into dataframes

In [ ]:
df_train = pd.read_csv('./data/train_dataset.csv')
df_val = pd.read_csv('./data/val_dataset.csv')
df_test = pd.read_csv('./data/test_dataset.csv')

# df_train = pd.read_csv('/kaggle/input/ai-2-dl-for-nlp-2025-homework-2/train_dataset.csv')
# df_val = pd.read_csv('/kaggle/input/ai-2-dl-for-nlp-2025-homework-2/val_dataset.csv')
# df_test = pd.read_csv('/kaggle/input/ai-2-dl-for-nlp-2025-homework-2/test_dataset.csv')

### Extract tweets and labels from the dataframes

In [ ]:
X_train, y_train = df_train['Text'], df_train['Label']

X_val, y_val = df_val['Text'], df_val['Label']

X_test = df_test['Text']

### Displaying the dimensions of dataframes

In [ ]:
print(f'Training dataset dimensions: {df_train.shape}')
print(f'Validate dataset dimensions: {df_val.shape}')
print(f'Test dataset dimensions: {df_test.shape}')

### Displaying metadata information about Train dataframe

In [ ]:
df_train.info()

### Displaying metadata information about Validation dataframe

In [ ]:
df_val.info()

### Checking the top 5 rows of the train set

In [ ]:
df_train.head()

### Checking the top 5 rows of the validation set

In [ ]:
df_val.head()

### We observe that train & validation datasets do not have any empty or duplicated rows

In [ ]:
print('Train Set')
display(df_train.isnull().sum())
display(df_train.duplicated().sum())

print('Validation Set')
display(df_val.isnull().sum())
display(df_val.duplicated().sum())

## Exploratory Data Analysis Before Text Preprocessing

### Plotting the distribution of sentiment categories in dataset. We notice that positive & negative labels are equally distributed

In [ ]:
labels_count = df_train['Label'].value_counts()
negative_tweets = labels_count[0]
positive_tweets = labels_count[1]

plot_pie_chart({'Negative': negative_tweets, 'Positive': positive_tweets}, 'Train Set Distribution of Categories')

In [ ]:
labels_count = df_val['Label'].value_counts()
negative_tweets = labels_count[0]
positive_tweets = labels_count[1]

plot_pie_chart({'Negative': negative_tweets, 'Positive': positive_tweets}, 'Validation Set Distribution of Categories')

### Creating a wordcloud using the tweet texts on train set before preprocessing

In [ ]:
create_wordcloud(df_train['Text'])

### Creating a wordcloud by using only the positive tweets before preprocessing

In [ ]:
create_wordcloud(df_train[df_train['Label'] == 1]['Text'])

### Creating a wordcloud by using only the negative tweets before preprocessing

In [ ]:
create_wordcloud(df_train[df_train['Label'] == 0]['Text'])

### Plotting a histogram with tweets length across the dataset

In [ ]:
tweets_length = get_tweets_length(df_train)
plot_histogram(tweets_length, 'Tweets Text Length', 'Number of Tweets', 'Tweets Length Before Preprocessing')

### Plotting minimum and maximum length of tweets on train set

In [ ]:
df_min_max = get_min_max_tweets(tweets_length)
plot_bar(list(df_min_max.columns), df_min_max.iloc[0, :].values.tolist(), 'Min & Max Tweets Text Length', 'Tweets Text length', 'Min & Max Tweets Length Before Text Preprocessing', False, False, True)

### Plotting the most common words (unigrams) of train dataset

In [ ]:
bar_plots_1ngrams = [res for res in get_top_tokens(df_train, top=10, n=1)] + ['Unigram symbols', 'Frequency']
plot_bar(bar_plots_1ngrams[0], bar_plots_1ngrams[1], bar_plots_1ngrams[2], bar_plots_1ngrams[3], 'Train Set, Unigrams Frequency Before Preprocessing', bary=True)

### Plotting the most common tokens (bigrams and trigrams) of train dataset

In [ ]:
bar_plots_2ngrams = [res for res in get_top_tokens(df_train, top=10, n=2)] + ['2 Words Phrases (2-grams)', 'Frequency']
bar_plots_3ngrams = [res for res in get_top_tokens(df_train, top=10, n=3)] + ['3 Words Phrases (3-grams)', 'Frequency']

subplot_bars(bar_plots_2ngrams, bar_plots_3ngrams, 'Train Set, Tweets Bigrams & Trigrams Before Text Preprocessing', bary=True)

### Plotting the number of emails and user mentions found in train dataset (Log Scale)

In [ ]:
emails_count = df_train.loc[df_train['Text'].str.contains(r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{1,}\b'), :].shape[0]

user_mentions_count = df_train.loc[df_train['Text'].str.contains(r'(?<![\w.])@\w+'), :].shape[0]

plot_bar(['Emails', 'Users Mentions'], [emails_count, user_mentions_count], '', '', 'Tweets Email and Username Mentions', False, True, True)

## NLP Text Preprocessing on Tweets

In [ ]:
nltk.download('punkt', quiet=True);
nltk.download('stopwords', quiet=True);
nltk.download('wordnet', quiet=True);

# !unzip /usr/share/nltk_data/corpora/wordnet.zip -d /usr/share/nltk_data/corpora/

### Slang word acronyms dictionary

In [ ]:
abbreviations = {
    "$" : " dollar ",
    "€" : " euro ",
    "4ao" : "for adults only",
    "a.m" : "before midday",
    "a3" : "anytime anywhere anyplace",
    "aamof" : "as a matter of fact",
    "acct" : "account",
    "adih" : "another day in hell",
    "afaic" : "as far as i am concerned",
    "afaict" : "as far as i can tell",
    "afaik" : "as far as i know",
    "afair" : "as far as i remember",
    "afk" : "away from keyboard",
    "app" : "application",
    "approx" : "approximately",
    "apps" : "applications",
    "asap" : "as soon as possible",
    "asl" : "age, sex, location",
    "atk" : "at the keyboard",
    "ave." : "avenue",
    "aymm" : "are you my mother",
    "ayor" : "at your own risk", 
    "b&b" : "bed and breakfast",
    "b+b" : "bed and breakfast",
    "b.c" : "before christ",
    "b2b" : "business to business",
    "b2c" : "business to customer",
    "b4" : "before",
    "b4n" : "bye for now",
    "b@u" : "back at you",
    "bae" : "before anyone else",
    "bak" : "back at keyboard",
    "bbbg" : "bye bye be good",
    "bbc" : "british broadcasting corporation",
    "bbias" : "be back in a second",
    "bbl" : "be back later",
    "bbs" : "be back soon",
    "be4" : "before",
    "bfn" : "bye for now",
    "blvd" : "boulevard",
    "bout" : "about",
    "brb" : "be right back",
    "bros" : "brothers",
    "brt" : "be right there",
    "bsaaw" : "big smile and a wink",
    "btw" : "by the way",
    "bwl" : "bursting with laughter",
    "c/o" : "care of",
    "cet" : "central european time",
    "cf" : "compare",
    "cia" : "central intelligence agency",
    "csl" : "can not stop laughing",
    "cu" : "see you",
    "cul8r" : "see you later",
    "cv" : "curriculum vitae",
    "cwot" : "complete waste of time",
    "cya" : "see you",
    "cyt" : "see you tomorrow",
    "dae" : "does anyone else",
    "dbmib" : "do not bother me i am busy",
    "diy" : "do it yourself",
    "dm" : "direct message",
    "dwh" : "during work hours",
    "e123" : "easy as one two three",
    "eet" : "eastern european time",
    "eg" : "example",
    "embm" : "early morning business meeting",
    "encl" : "enclosed",
    "encl." : "enclosed",
    "etc" : "and so on",
    "faq" : "frequently asked questions",
    "fawc" : "for anyone who cares",
    "fb" : "facebook",
    "fc" : "fingers crossed",
    "fig" : "figure",
    "fimh" : "forever in my heart", 
    "ft." : "feet",
    "ft" : "featuring",
    "ftl" : "for the loss",
    "ftw" : "for the win",
    "fwiw" : "for what it is worth",
    "fyi" : "for your information",
    "g9" : "genius",
    "gahoy" : "get a hold of yourself",
    "gal" : "get a life",
    "gcse" : "general certificate of secondary education",
    "gfn" : "gone for now",
    "gg" : "good game",
    "gl" : "good luck",
    "glhf" : "good luck have fun",
    "gmt" : "greenwich mean time",
    "gmta" : "great minds think alike",
    "gn" : "good night",
    "g.o.a.t" : "greatest of all time",
    "goat" : "greatest of all time",
    "goi" : "get over it",
    "gps" : "global positioning system",
    "gr8" : "great",
    "gratz" : "congratulations",
    "gyal" : "girl",
    "h&c" : "hot and cold",
    "hp" : "horsepower",
    "hr" : "hour",
    "hrh" : "his royal highness",
    "ht" : "height",
    "ibrb" : "i will be right back",
    "ic" : "i see",
    "icq" : "i seek you",
    "icymi" : "in case you missed it",
    "idc" : "i do not care",
    "idgadf" : "i do not give a damn fuck",
    "idgaf" : "i do not give a fuck",
    "idk" : "i do not know",
    "ie" : "that is",
    "i.e" : "that is",
    "ifyp" : "i feel your pain",
    "IG" : "instagram",
    "iirc" : "if i remember correctly",
    "ilu" : "i love you",
    "ily" : "i love you",
    "imho" : "in my humble opinion",
    "imo" : "in my opinion",
    "imu" : "i miss you",
    "iow" : "in other words",
    "irl" : "in real life",
    "j4f" : "just for fun",
    "jic" : "just in case",
    "jk" : "just kidding",
    "jsyk" : "just so you know",
    "l8r" : "later",
    "lb" : "pound",
    "lbs" : "pounds",
    "ldr" : "long distance relationship",
    "lmao" : "laugh my ass off",
    "lmfao" : "laugh my fucking ass off",
    "lol" : "laughing out loud",
    "ltd" : "limited",
    "ltns" : "long time no see",
    "m8" : "mate",
    "mf" : "motherfucker",
    "mfs" : "motherfuckers",
    "mfw" : "my face when",
    "mofo" : "motherfucker",
    "mph" : "miles per hour",
    "mr" : "mister",
    "mrw" : "my reaction when",
    "ms" : "miss",
    "mte" : "my thoughts exactly",
    "nagi" : "not a good idea",
    "nbc" : "national broadcasting company",
    "nbd" : "not big deal",
    "nfs" : "not for sale",
    "ngl" : "not going to lie",
    "nhs" : "national health service",
    "nrn" : "no reply necessary",
    "nsfl" : "not safe for life",
    "nsfw" : "not safe for work",
    "nth" : "nice to have",
    "nvr" : "never",
    "nyc" : "new york city",
    "oc" : "original content",
    "og" : "original",
    "ohp" : "overhead projector",
    "oic" : "oh i see",
    "omdb" : "over my dead body",
    "omg" : "oh my god",
    "omw" : "on my way",
    "p.a" : "per annum",
    "p.m" : "after midday",
    "pm" : "prime minister",
    "poc" : "people of color",
    "pov" : "point of view",
    "pp" : "pages",
    "ppl" : "people",
    "prw" : "parents are watching",
    "ps" : "postscript",
    "pt" : "point",
    "ptb" : "please text back",
    "pto" : "please turn over",
    "qpsa" : "what happens", #"que pasa",
    "ratchet" : "rude",
    "rbtl" : "read between the lines",
    "rlrt" : "real life retweet", 
    "rofl" : "rolling on the floor laughing",
    "roflol" : "rolling on the floor laughing out loud",
    "rotflmao" : "rolling on the floor laughing my ass off",
    "rt" : "retweet",
    "ruok" : "are you ok",
    "sfw" : "safe for work",
    "sk8" : "skate",
    "smh" : "shake my head",
    "sq" : "square",
    "srsly" : "seriously", 
    "ssdd" : "same stuff different day",
    "tbh" : "to be honest",
    "tbs" : "tablespooful",
    "tbsp" : "tablespooful",
    "tfw" : "that feeling when",
    "thks" : "thank you",
    "tho" : "though",
    "thx" : "thank you",
    "tia" : "thanks in advance",
    "til" : "today i learned",
    "tl;dr" : "too long i did not read",
    "tldr" : "too long i did not read",
    "tmb" : "tweet me back",
    "tntl" : "trying not to laugh",
    "ttyl" : "talk to you later",
    "u" : "you",
    "u2" : "you too",
    "u4e" : "yours for ever",
    "utc" : "coordinated universal time",
    "w/" : "with",
    "w/o" : "without",
    "w8" : "wait",
    "wassup" : "what is up",
    "wb" : "welcome back",
    "wtf" : "what the fuck",
    "wtg" : "way to go",
    "wtpa" : "where the party at",
    "wuf" : "where are you from",
    "wuzup" : "what is up",
    "wywh" : "wish you were here",
    "yd" : "yard",
    "ygtr" : "you got that right",
    "ynk" : "you never know",
    "zzz" : "sleeping bored and tired"
}

In [ ]:
def convert_abbrev(word):
    return abbreviations[word] if word in abbreviations.keys() else word

### Text Preprocessing is done on two phases to determine which is the best configuration for out dataset
- Preprocessing with stop words filtering
- Preprocessing with stop words preservation

We also test slang words conversion to our model

### The `preprocess_text` function applies the following preprocessing steps:
- **Lowercasing**: Converts all characters to lowercase.
- **Expanding Contractions**: Expanding combinations of words that are shortened.
- **Removing hashtags**: Removing hashtag symbols, leaving the words intact.
- **Removing Mentions**: Username mentions in tweets are removed.
- **Removing Email Addresses**: Email addresses are removed'.
- **Covert Slang Words** *(if enabled)*: If 'convert_slang' is true then filter slang abbreviations.
- **Filtering Stop Words** *(if enabled)*: If 'filter_stopwords' is true then stop words are removed from tweets.

**URL and digits removal were tested and produced lower accuracy to our models in this homework**

In [ ]:
tweet_tokenizer = TweetTokenizer()

def preprocess_text(text, convert_slang=False, filter_stopwords=False):
    text = text.lower()
    text = contractions.fix(text)
    
    text = re.sub(r'#', '', text)
    text = re.sub(r'(?<![\w.])@\w+', '', text)
    text = re.sub(r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\b', '', text)
    
    tokens = tweet_tokenizer.tokenize(text)
    
    if convert_slang:
        tokens = [convert_abbrev(word) for word in tokens]
    
    if filter_stopwords:
        stop_words = set(stopwords.words('english'))
        tokens = [word for word in tokens if word not in stop_words]

    return ' '.join(tokens)

### Removing rows with zero text length (after doing preprocessing) **only** on train and validation datasets

In [ ]:
def filter_non_empty(df):
    return df[df['Text'].str.split().str.len() != 0]

### Function `clean_dataset` does the following steps:
- Creates a copy of the dataframe.
- Removes non-ascii characters from tweets.
- Runs the `preprocess_text` function to further preprocess the tweets (explained above).
- Removes any row with zero text length (on train and validation set only).

In [ ]:
def clean_dataset(df, filter_stopwords=False, filter_rows=True, convert_slang=False):
    df_clean = df.copy()
    df_clean['Text'] = df_clean['Text'].str.encode('ascii', 'ignore').str.decode('ascii')
    df_clean['Text'] = df_clean['Text'].apply(preprocess_text, filter_stopwords=filter_stopwords, convert_slang=convert_slang)
    return filter_non_empty(df_clean) if filter_rows else df_clean

### Running tweets preprocessing on train, validation and test set with filtering stop words and slang conversion activated

In [ ]:
df_train_cln = clean_dataset(df_train, filter_stopwords=True, convert_slang=True)
df_val_cln = clean_dataset(df_val, filter_stopwords=True, convert_slang=True)
df_test_cln = clean_dataset(df_test, filter_stopwords=True, filter_rows=False, convert_slang=True)

X_train_prep, y_train_prep = df_train_cln['Text'], df_train_cln['Label']
X_val_prep, y_val_prep   = df_val_cln['Text'], df_val_cln['Label']
X_test_prep = df_test_cln['Text']

### Displaying the dimenstions of train, validation and test dataset matrices

In [ ]:
print(f"X train Preprocessed Dimensions: {X_train_prep.shape}")
print(f"y train Preprocessed Dimensions: {y_train_prep.shape}")

print(f"X validation Preprocessed Dimensions: {X_val_prep.shape}")
print(f"y validation Preprocessed Dimensions: {y_val_prep.shape}")

print(f"X test Preprocessed Dimensions: {X_test_prep.shape}")

### Checking the tweets after running preprocessing

In [ ]:
df_train_cln.head()

In [ ]:
df_val_cln.head()

## Exploratory Data Analysis After Running Text Preprocessing (With Stop Words Filtering and slang words conversion)

### Plotting a histogram with tweets length across the training dataset after preprocessing

In [ ]:
tweets_length_clean = get_tweets_length(df_train_cln)
plot_histogram(tweets_length_clean, 'Tweets Text Length', 'Number of Tweets', 'Tweets Length After Text Preprocessing')

### Plotting minimum and maximum length of tweets after running preprocessing

In [ ]:
df_min_max = get_min_max_tweets(tweets_length_clean)
plot_bar(list(df_min_max.columns), df_min_max.iloc[0, :].values.tolist(), 'Min & Max Tweets Length', 'Tweets Text Length', 'Min & Max Tweets Length After Text Preprocessing', False, False, True)

### Creating a wordcloud using the tweet texts on train set after running preprocessing

In [ ]:
create_wordcloud(df_train_cln['Text'])

### Creating a wordcloud by using only the positive tweets after running preprocessing

In [ ]:
create_wordcloud(df_train_cln[df_train_cln['Label'] == 1]['Text'])

### Creating a wordcloud by using only the negative tweets after running preprocessing

In [ ]:
create_wordcloud(df_train_cln[df_train_cln['Label'] == 0]['Text'])

### Plotting the most common words (unigrams) of train dataset after running preprocessing

In [ ]:
bar_plots_1ngrams = [res for res in get_top_tokens(df_train_cln, top=10, n=1)] + ['Unigram Symbols', 'Frequency']
plot_bar(bar_plots_1ngrams[0], bar_plots_1ngrams[1], bar_plots_1ngrams[2], bar_plots_1ngrams[3], 'Train Set, Unigrams Frequency After Text Preprocessing', bary=True)

### Plotting the most common tokens (bigrams and trigrams) of train dataset after running preprocessing

In [ ]:
bar_plots_2ngrams = [res for res in get_top_tokens(df_train_cln, top=10, n=2)] + ['2 Words Phrases (2-grams)', 'Frequency']
bar_plots_3ngrams = [res for res in get_top_tokens(df_train_cln, top=10, n=3)] + ['3 Words Phrases (3-grams)', 'Frequency']

subplot_bars(bar_plots_2ngrams, bar_plots_3ngrams, 'Train Set, Tweets Bigrams & Trigrams After Preprocessing', bary=True)

In [ ]:
bar_plots_1ngrams = [res for res in get_top_tokens(df_val_cln, top=10, n=1)] + ['Unigram Symbols', 'Frequency']
plot_bar(bar_plots_1ngrams[0], bar_plots_1ngrams[1], bar_plots_1ngrams[2], bar_plots_1ngrams[3], 'Validation Set, Unigrams Frequency After Text Preprocessing', bary=True)

In [ ]:
bar_plots_2ngrams = [res for res in get_top_tokens(df_val_cln, top=10, n=2)] + ['2 Words Phrases (2-grams)', 'Frequency']
bar_plots_3ngrams = [res for res in get_top_tokens(df_val_cln, top=10, n=3)] + ['3 Words Phrases (3-grams)', 'Frequency']

subplot_bars(bar_plots_2ngrams, bar_plots_3ngrams, 'Validation Set, Tweets Bigrams & Trigrams After Preprocessing', bary=True)

## Feature Extraction: Word2Vec using GoogleNews pretrained vector embeddings

#### The acronyms used on preprocessing experimentations are the following: 

#### - **PRFS**: Preprocessing & Filtering Stopwords
#### - **PRPS**: Preprocessing & Preserving Stopwords

#### **Converting slang acronyms to original tweets is also tested**

### Function that calculates **Accuracy**, **Precission**, **Recall** and **F1 scores** and returns a dictionary object

In [ ]:
def calc_model_score(y_true, y_pred):
    metrics = {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1 Score": f1_score(y_true, y_pred, zero_division=0)
    }
    return metrics

### Function that plots the final scores (accuracy, precision, recall, F1) of a BERT model on the best epoch step

In [ ]:
def plot_best_scores(results, title):
    keys = ['best_epoch_accuracy', 'best_epoch_precision', 'best_epoch_recall', 'best_epoch_f1']
    label_mapping = {
        'best_epoch_accuracy': 'Accuracy',
        'best_epoch_precision': 'Precision',
        'best_epoch_recall': 'Recall',
        'best_epoch_f1': 'F1'
    }
    colors_list = ['#1C77C3', '#F34213', '#E0CA3C', '#3E2F5B']
    
    scores = [results[key] * 100 for key in keys]
    labels = [label_mapping[key] for key in keys]

    _, ax = plt.subplots(figsize=(8,6))
    bars = ax.bar(labels, scores, color=colors_list, edgecolor="black")

    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, height + 0.5, f"{height:.2f}%", ha='center', va='bottom')
    
    ax.set_ylabel("Score (%)")
    ax.set_title(title)
    ax.set_ylim(0, 110)
    
    plt.tight_layout()
    plt.show()

### Function that plots BERT model scores & training, validation losses for each epoch

In [ ]:
def plot_epochs_scores(results, title):
    epochs_number = len(results['Validation Loss'])
    epochs = list(range(1, epochs_number + 1))
    xticks = [x for x in epochs if x == 1 or x % max(1, epochs_number // 10) == 0]
    
    grouped = {
        'Loss': ['Train Loss', 'Validation Loss'],
        'Accuracy': ['Train Accuracy', 'Validation Accuracy']
    }

    merged_keys = grouped['Loss'] + grouped['Accuracy']
    single_metrics = [k for k in results if k not in merged_keys]

    fig = plt.figure(figsize=(15, 8))
    fig.suptitle(title, fontsize=16)
    grid_spec = gridspec.GridSpec(2, 1)

    gs_top = gridspec.GridSpecFromSubplotSpec(1, 2, grid_spec[0], wspace=0.3)
    for ax, (group_name, keys) in zip([fig.add_subplot(gs_top[0, 0]), fig.add_subplot(gs_top[0, 1])], grouped.items()):
        for key in keys:
            ax.plot(epochs, results[key], marker='o', label=key)
        ax.set_xticks(xticks)
        ax.set_xlim(0.5, epochs_number + 0.5)
        ax.margins(x=0.05)
        ax.set_title(group_name, fontsize=12)
        ax.set_xlabel('Epoch', fontsize=10)
        ax.set_ylabel(group_name, fontsize=10)
        ax.grid(True)
        ax.legend(fontsize=8)

    gs_bottom = gridspec.GridSpecFromSubplotSpec(1, 3, grid_spec[1], wspace=0.3)
    for i, key in enumerate(single_metrics[:3]):
        ax = fig.add_subplot(gs_bottom[0, i])
        ax.plot(epochs, results[key], marker='o')
        ax.set_xticks(xticks)
        ax.set_xlim(0.5, epochs_number + 0.5)
        ax.margins(x=0.05)
        ax.set_title(key, fontsize=12)
        ax.set_xlabel('Epoch', fontsize=10)
        ax.set_ylabel(key, fontsize=10)
        ax.grid(True)

    plt.tight_layout()
    plt.show()

### Function that calculates the ROC curve

In [ ]:
def calc_roc_curve(y, y_pred):
    fpr, tpr, _ = roc_curve(y, y_pred)
    roc_auc = auc(fpr, tpr)
    return fpr, tpr, roc_auc

### Function that plots the ROC curve

In [ ]:
def plot_roc_curve(fpr, tpr, roc_auc, estimator_name):
    display = RocCurveDisplay(fpr=fpr, tpr=tpr, roc_auc=roc_auc, estimator_name=estimator_name)
    display.plot()
    plt.show()

### Function that calculates the confusion matrix, given the classification labels

In [ ]:
def calc_confusion_matrix(y, y_pred):
    return confusion_matrix(y, y_pred)

### Function that plots the confusion matrix

In [ ]:
def plot_confusion_matrix(cm):
    disp = ConfusionMatrixDisplay(confusion_matrix=cm)
    disp.plot()
    plt.show()

### Loading GoogleNews pretrained vectos in a Word2Vec format. The dimensions of the embeddings are 300

In [ ]:
embeddings_dim = 300
word2vec_google = KeyedVectors.load_word2vec_format('GoogleNews-vectors-negative300.bin', binary=True)

### Function that returns the best results based on the maximum accuracy

In [ ]:
def get_best_results(results):
    return max(results, key=lambda c: c[1]['best_epoch_accuracy'])

### Creating a seeded random sampler using torch Generators, to be used in Dataloaders

In [ ]:
def get_random_sampler(dataset, seed):
    generator = torch.Generator()
    generator.manual_seed(seed)
    return RandomSampler(dataset, generator=generator)

### Function that trains BERT model. It is used to train our model for epochs and calculated the average train set loss

In [ ]:
def train_epoch(model, dataloader, optimizer, device, scheduler=None):
    model.train()
    running_loss = 0.0

    for batch in dataloader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        optimizer.zero_grad()

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss

        loss.backward()
        optimizer.step()

        if scheduler:
            scheduler.step()

        running_loss += loss.item()

    epoch_loss = running_loss / len(dataloader)
    return epoch_loss

### Function that evaluates the BERT model, calculates the validation set loss and all the remaining model scores

In [ ]:
def evaluate_model(model, dataloader, device):
    model.eval()
    y_pred_list = []
    y_true_list = []
    total_loss = 0.0

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            logits = outputs.logits

            total_loss += loss.item()

            predictions = torch.argmax(logits, dim=-1)

            y_pred_list.append(predictions.cpu().numpy())
            y_true_list.append(labels.cpu().numpy())

    y_pred_list = np.concatenate(y_pred_list)
    y_true_list = np.concatenate(y_true_list)

    avg_loss = total_loss / len(dataloader)
    metrics = calc_model_score(y_true_list, y_pred_list)
    metrics['loss'] = avg_loss

    return metrics

### Function that prints BERTS' performance evaluation score including train and validation losses

In [ ]:
def print_epoch_scores(print_res, epoch, n_epochs, train_loss, train_metrics , val_metrics):
    if epoch % print_res == 0:
        print(f"Epoch {epoch}/{n_epochs} | "
            f"Train Loss: {train_loss:.4f} | Val Loss: {val_metrics['loss']:.4f} | "
            f"Train Acc: {train_metrics['Accuracy']:.4f} | "
            f"Val Acc: {val_metrics['Accuracy']:.4f} | "
            f"Val Precision: {val_metrics['Precision']:.4f} | "
            f"Val Recall: {val_metrics['Recall']:.4f} | "
            f"Val F1: {val_metrics['F1 Score']:.4f}"
        )

### Function that trains a BERT model over multiple epochs, evaluates performance on training and validation sets and also supports early stopping. Stores model scores such as loss, accuracy, precision, recall, and F1 score for each epoch

In [ ]:
def train_model(model, train_loader, val_loader, optimizer, n_epochs, device, scheduler=None, early=False, print_res=1):
    results = {
        'Train Loss': [],
        'Validation Loss': [],
        'Train Accuracy': [],
        'Validation Accuracy': [],
        'Validation Precision': [],
        'Validation Recall': [],
        'Validation F1 Score': [],
        'Best Epoch Number': 1
    }

    no_loss_improve = 0
    best_val_loss = float('inf')
    best_state = None

    for epoch in range(1, n_epochs + 1):
        train_loss = train_epoch(model, train_loader, optimizer, device=device, scheduler=scheduler)

        train_metrics = evaluate_model(model, train_loader, device=device)
        val_metrics = evaluate_model(model, val_loader, device=device)

        results['Train Loss'].append(train_loss)
        results['Validation Loss'].append(val_metrics['loss'])
        results['Train Accuracy'].append(train_metrics['Accuracy'])
        results['Validation Accuracy'].append(val_metrics['Accuracy'])
        results['Validation Precision'].append(val_metrics['Precision'])
        results['Validation Recall'].append(val_metrics['Recall'])
        results['Validation F1 Score'].append(val_metrics['F1 Score'])

        print_epoch_scores(print_res, epoch, n_epochs, train_loss, train_metrics , val_metrics)

        if val_metrics['loss'] < best_val_loss:
            best_val_loss = val_metrics['loss']
            no_loss_improve = 0
            best_epoch = epoch
            results['best_epoch_val_loss'] = best_val_loss
            results['best_epoch_accuracy'] = val_metrics['Accuracy']
            results['best_epoch_precision'] = val_metrics['Precision']
            results['best_epoch_recall'] = val_metrics['Recall']
            results['best_epoch_f1'] = val_metrics['F1 Score']
            results['Best Epoch Number'] = best_epoch
            best_state = model.state_dict()
        else:
            no_loss_improve += 1

        if early and no_loss_improve >= 2:
            print(f"Early stopping triggered after {epoch} epochs.")
            print(f"Best model found at epoch {best_epoch} with validation loss {best_val_loss:.4f}")
            break

    return results, best_state

### Function that evaluates the BERT model on a dataset and returns the predicted classification labels and probabilities

In [ ]:
def evaluate(model, dataset, device):
    input_ids = dataset['input_ids']
    attention_mask = torch.tensor(dataset['attention_mask'])

    test_dataset = TensorDataset(input_ids, attention_mask)
    test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

    preds = []
    probs = []
    
    model.eval()
    with torch.no_grad():
        for input_ids, attention_mask in test_loader:
            input_ids, attention_mask = input_ids.to(device), attention_mask.to(device)
            logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
            softmax_probs = torch.softmax(logits, dim=1)
            preds.extend(torch.argmax(softmax_probs, dim=1).cpu().numpy())
            probs.extend(softmax_probs[:, 1].cpu().numpy())

    return np.array(preds), np.array(probs)

### Function that creates tweets datasets using the bert tokenizer and returns input IDs, attention mask and labels

In [ ]:
class TweetDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            padding='max_length',
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt",
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long),
        }

### Function that samples a fraction of the training and validation dataset (using a seed) and returns the datasets, ready to be used on a dataloader

In [ ]:
def create_sampled_datasets(X_train_prep, y_train_prep, X_val_prep, y_val_prep, tokenizer, seed, sample_frac, max_length):
    train_sample = X_train_prep.sample(frac=sample_frac, random_state=seed)
    y_train_sample = y_train_prep.loc[train_sample.index]

    val_sample = X_val_prep.sample(frac=sample_frac, random_state=seed)
    y_val_sample = y_val_prep.loc[val_sample.index]

    train_sample = train_sample.reset_index(drop=True)
    y_train_sample = y_train_sample.reset_index(drop=True)

    val_sample = val_sample.reset_index(drop=True)
    y_val_sample = y_val_sample.reset_index(drop=True)

    train_dataset_sample = TweetDataset(train_sample.tolist(), y_train_sample.tolist(), tokenizer, max_length=max_length)
    val_dataset_sample = TweetDataset(val_sample.tolist(), y_val_sample.tolist(), tokenizer, max_length=max_length)

    return train_dataset_sample, val_dataset_sample

### Selecting GPU device for pytorch else CPU

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

### Loading the BERT tokenizer

In [ ]:
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

In [ ]:
preprocess_configs = [
    {"filter_stopwords": True, "convert_slang": False},
    {"filter_stopwords": False, "convert_slang": False},
    {"filter_stopwords": False, "convert_slang": True}
]

preprocess_results = []

### Preprocessing tweets text with filtering stop words (PRFS) & slang words conversion disabled

In [ ]:
df_train_cln = clean_dataset(df_train, **preprocess_configs[0])
df_val_cln = clean_dataset(df_val, **preprocess_configs[0])
df_test_cln = clean_dataset(df_test, **preprocess_configs[0], filter_rows=False)

X_train_prep, y_train_prep = df_train_cln['Text'], df_train_cln['Label']
X_val_prep, y_val_prep = df_val_cln['Text'], df_val_cln['Label']
X_test_prep = df_test_cln['Text']

### Creating the train and validation set dataloaders by sampling (using specific seed) the 25% of datasets to experiment with BERTS' hyperparameters

In [ ]:
train_dataset_sample, val_dataset_sample = create_sampled_datasets(X_train_prep, y_train_prep, X_val_prep, y_val_prep, tokenizer, seed=seed, sample_frac=1/4, max_length=128)

train_sample_loader = DataLoader(train_dataset_sample, batch_size=32, sampler=get_random_sampler(train_dataset_sample, seed))
val_sample_loader = DataLoader(val_dataset_sample, batch_size=64, shuffle=False)

### Loading the pretrained BERT model and running 3 epochs using the PRFS & no slang words conversion dataset

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)
model.to(device)

optimizer = AdamW(model.parameters(), lr=2e-5)

results, best_state = train_model(model, train_sample_loader, val_sample_loader, optimizer, n_epochs=3, device=device)

preprocess_results.append((preprocess_configs[0], results))

plot_epochs_scores(results, f"BERT, PRFS, Slang Conversion Disabled, Optimizer: AdamW, lr: 2e-5, 3 Epochs")
plot_best_scores(results, f"BERT, PRFS, Slang Conversion Disabled, Best Validation Set Score\nOptimizer: AdamW, lr: 2e-5, 3 Epochs")

### Preprocessing tweets text without filtering stop words (PRPS) & slang words conversion disabled

In [ ]:
df_train_cln = clean_dataset(df_train, **preprocess_configs[1])
df_val_cln = clean_dataset(df_val, **preprocess_configs[1])
df_test_cln = clean_dataset(df_test, **preprocess_configs[1], filter_rows=False)

X_train_prep, y_train_prep = df_train_cln['Text'], df_train_cln['Label']
X_val_prep, y_val_prep = df_val_cln['Text'], df_val_cln['Label']
X_test_prep = df_test_cln['Text']

### Creating the train and validation set dataloaders by sampling (using specific seed) the 25% of datasets to experiment with BERTS' hyperparameters

In [ ]:
train_dataset_sample, val_dataset_sample = create_sampled_datasets(X_train_prep, y_train_prep, X_val_prep, y_val_prep, tokenizer, seed=seed, sample_frac=1/4, max_length=128)

train_sample_loader = DataLoader(train_dataset_sample, batch_size=32, sampler=get_random_sampler(train_dataset_sample, seed))
val_sample_loader = DataLoader(val_dataset_sample, batch_size=64, shuffle=False)

### Loading the pretrained BERT model and running 3 epochs using the PRPS & no slang words conversion dataset

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)
model.to(device)

optimizer = AdamW(model.parameters(), lr=2e-5)

results, best_state = train_model(model, train_sample_loader, val_sample_loader, optimizer, n_epochs=3, device=device)

preprocess_results.append((preprocess_configs[1], results))

plot_epochs_scores(results, f"BERT, PRPS, Slang Conversion Disabled, Optimizer: AdamW, lr: 2e-5, 3 Epochs")
plot_best_scores(results, f"BERT, PRPS, Slang Conversion Disabled, Best Validation Set Score\nOptimizer: AdamW, lr: 2e-5, 3 Epochs")

### Preprocessing tweets text without filtering stop words (PRPS) & slang words conversion enabled

In [ ]:
df_train_cln = clean_dataset(df_train, **preprocess_configs[2])
df_val_cln = clean_dataset(df_val, **preprocess_configs[2])
df_test_cln = clean_dataset(df_test, **preprocess_configs[2], filter_rows=False)

X_train_prep, y_train_prep = df_train_cln['Text'], df_train_cln['Label']
X_val_prep, y_val_prep = df_val_cln['Text'], df_val_cln['Label']
X_test_prep = df_test_cln['Text']

### Creating the train and validation set dataloaders by sampling (using specific seed) the 25% of datasets to experiment with BERTS' hyperparameters

In [ ]:
train_dataset_sample, val_dataset_sample = create_sampled_datasets(X_train_prep, y_train_prep, X_val_prep, y_val_prep, tokenizer, seed=seed, sample_frac=1/4, max_length=128)

train_sample_loader = DataLoader(train_dataset_sample, batch_size=32, sampler=get_random_sampler(train_dataset_sample, seed))
val_sample_loader = DataLoader(val_dataset_sample, batch_size=64, shuffle=False)

### Loading the pretrained BERT model and running 3 epochs using the PRPS with slang words conversion enabled dataset

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)
model.to(device)

optimizer = AdamW(model.parameters(), lr=2e-5)

results, best_state = train_model(model, train_sample_loader, val_sample_loader, optimizer, n_epochs=3, device=device)

preprocess_results.append((preprocess_configs[2], results))

plot_epochs_scores(results, f"BERT, PRPS, Slang Conversion Enabled, Optimizer: AdamW, lr: 2e-5, 3 Epochs")
plot_best_scores(results, f"BERT, PRPS, Slang Conversion Enabled, Best Validation Set Score\nOptimizer: AdamW, lr: 2e-5, 3 Epochs")

### Preprocessing again the initial datasets using the best preprocessing configuration based on the best accuracy on the lowest validation loss epoch

In [ ]:
best_config, best_results = max(preprocess_results, key=lambda x: x[1]['best_epoch_accuracy'])
print(f"Best preprocessing configuration: {best_config}")

In [ ]:
df_train_cln = clean_dataset(df_train, **best_config)
df_val_cln = clean_dataset(df_val, **best_config)
df_test_cln = clean_dataset(df_test, **best_config, filter_rows=False)

X_train_prep, y_train_prep = df_train_cln['Text'], df_train_cln['Label']
X_val_prep, y_val_prep = df_val_cln['Text'], df_val_cln['Label']
X_test_prep = df_test_cln['Text']

### Checking maximum length of tweets when encoded, before truncation

In [ ]:
token_lengths = [len(tokenizer.encode(sent, add_special_tokens=True)) for sent in X_train_prep]
max_len = max(token_lengths)
print("Max tweet length:", max_len)

plt.hist(token_lengths, bins=50)
plt.title("Tokenized tweet Lengths")
plt.xlabel("Token count")
plt.ylabel("Frequency")
plt.show()

### Experimenting with BERT tokenizer's embeddings max length

In [ ]:
tokenizer_configs = {
    'Max Length 64': 64,
    'Max Length 128': 128,
    'Max Length 256': 256
}

### Loading the pretrained BERT model and running 3 epochs using the best preprocessing conf dataset on multiple token max lengths

In [ ]:
tokenizer_results = {}

for config_name, max_len in tokenizer_configs.items():

    train_dataset_tok, val_dataset_tok = create_sampled_datasets(
        X_train_prep, 
        y_train_prep,
        X_val_prep, 
        y_val_prep,
        tokenizer, 
        seed=seed, 
        sample_frac=1/4,
        max_length=max_len
    )

    train_loader = DataLoader(train_dataset_tok, batch_size=32, sampler=get_random_sampler(train_dataset_tok, seed))
    val_loader = DataLoader(val_dataset_tok, batch_size=64, shuffle=False)

    model = AutoModelForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)
    model.to(device)

    optimizer = AdamW(model.parameters(), lr=2e-5)

    results, best_state = train_model(model, train_loader, val_loader, optimizer=optimizer, n_epochs=3, device=device)

    tokenizer_results[config_name] = results

    plot_epochs_scores(results, f"BERT, Optimizer: AdamW, lr: 2e-5, 3 Epochs, Tokens Max Length: {max_len}")
    plot_best_scores(results, f"BERT, Best Validation Set Score\nOptimizer: AdamW, lr: 2e-5, 3 Epochs, Tokens Max Length: {max_len}")

### Retrieving the max length configuration with the BERT model that had the highest validation set accuracy with the lowest validation loss on an epoch step

In [ ]:
best_length_name, best_result = get_best_results(list(tokenizer_results.items())[:2])
best_val_acc = best_result['best_epoch_accuracy']

print(f"Best Tokens Max Length: {best_length_name} with validation accuracy: {best_val_acc:.4f}")

best_max_length = tokenizer_configs[best_length_name]

best_result

### Continuing using the best BERT tokenizer max length

In [ ]:
train_dataset_sample, val_dataset_sample = create_sampled_datasets(X_train_prep, y_train_prep, X_val_prep, y_val_prep, tokenizer, seed=seed, sample_frac=1/4, max_length=best_max_length)

train_sample_loader = DataLoader(train_dataset_sample, batch_size=32, sampler=get_random_sampler(train_dataset_sample, seed))
val_sample_loader = DataLoader(val_dataset_sample, batch_size=64, shuffle=False)

### Experimenting with the following optimizers based on validation set accuracy on the lowest validation set loss

In [ ]:
optimizers = {
    "Adam": Adam,
    "Nadam": NAdam,
    "RAdam": RAdam,
    "RMSprop": RMSprop
}

### Loading the pretrained BERT model and running 3 epochs using the best preprocessing config dataset with the above optimizers

In [ ]:
opt_results = {'AdamW': best_result}

for opt_name, opt_class in optimizers.items():

    model = AutoModelForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)
    model.to(device)

    optimizer = opt_class(model.parameters(), lr=2e-5)

    results, best_state = train_model(model, train_sample_loader, val_sample_loader, optimizer, n_epochs=3, device=device)
    
    opt_results[opt_name] = results

    plot_epochs_scores(results, f"BERT, Optimizer: {opt_name}, lr: 2e-5, 3 Epochs")
    plot_best_scores(results, f"BERT, Best Validation Set Score\nOptimizer: {opt_name}, lr: 2e-5, 3 Epochs")

### Retrieving the best optimizer configuration with the BERT model that had the highest validation set accuracy with the lowest validation loss on an epoch step

In [ ]:
optimizers['AdamW'] = AdamW
best_opt_name, best_result = get_best_results(opt_results.items())
best_val_acc = best_result['best_epoch_accuracy']

print(f"Best optimizer: {best_opt_name} with validation accuracy: {best_val_acc:.4f}")

best_optimizer = optimizers[best_opt_name]

best_result

### Using the best optimizer, conducting an experiment using an optimizer scheduler during training

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)
model.to(device)

optimizer = best_optimizer(model.parameters(), lr=2e-5)

total_steps = len(train_sample_loader) * 3 
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0, 
    num_training_steps=total_steps
)

results, best_state = train_model(model, train_sample_loader, val_sample_loader, optimizer, scheduler=scheduler, n_epochs=3, device=device)

opt_results[opt_name] = results

plot_epochs_scores(results, f"BERT, Optimizer: {opt_name}, Scheduler Enabled, lr: 2e-5, 3 Epochs")
plot_best_scores(results, f"BERT, Best Validation Set Score\nOptimizer: {opt_name}, Scheduler Enabled, lr: 2e-5, 3 Epochs")

In [ ]:
use_scheduler = False if best_val_acc > results['best_epoch_accuracy'] else True
use_scheduler

### Running Optuna to search for more optimized hyperparameters for learning rate, batrch size and weight decay, minimizig the model's validation loss

In [ ]:
def objective(trial):
    lr = trial.suggest_float('lr', 1e-5, 5e-5, log=True)
    batch_size = trial.suggest_categorical('batch_size', [16, 32])
    weight_decay = trial.suggest_float('weight_decay', 0.0, 0.1, step=0.01)
    eps = trial.suggest_float('eps', 1e-8, 1e-6, log=True)

    train_sample_loader = DataLoader(train_dataset_sample, batch_size=batch_size, sampler=get_random_sampler(train_dataset_sample, seed))
    val_sample_loader = DataLoader(val_dataset_sample, batch_size=64, shuffle=False)

    model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)
    model.to(device)

    optimizer = best_optimizer(model.parameters(), lr=lr, weight_decay=weight_decay, eps=eps)

    scheduler = None
    if use_scheduler:
        total_steps = len(train_sample_loader) * 2
        scheduler = get_linear_schedule_with_warmup(
            optimizer,
            num_warmup_steps=0,
            num_training_steps=total_steps
        )

    results, _ = train_model(
        model,
        train_sample_loader,
        val_sample_loader,
        optimizer,
        scheduler=scheduler,
        n_epochs=2,
        device=device,
        print_res=50
    )

    return results['best_epoch_val_loss']

### Starting by using the best model hyperparameters thus far

In [ ]:
sampler = optuna.samplers.TPESampler(seed=seed)

study = optuna.create_study(direction='minimize', sampler=sampler)

study.enqueue_trial({
    'lr': 2e-5,
    'batch_size': 32,
    'weight_decay': 0.0,
    'eps': 1e-8
})

study.optimize(objective, n_trials=11)

### Retrieving optuna's best hyperparameters in order to train the model on the entire dataset

In [ ]:
best_params = study.best_params

final_weight_decay = best_params['weight_decay']
final_batch_size = best_params['batch_size']
final_lr = best_params['lr']
final_eps = best_params['eps']

In [ ]:
X_train_prep = X_train_prep.reset_index(drop=True)
y_train_prep = y_train_prep.reset_index(drop=True)
X_val_prep = X_val_prep.reset_index(drop=True)
y_val_prep = y_val_prep.reset_index(drop=True)

In [ ]:
print(f"X train Preprocessed Dimensions: {X_train_prep.shape}")
print(f"y train Preprocessed Dimensions: {y_train_prep.shape}")

print(f"X validation Preprocessed Dimensions: {X_val_prep.shape}")
print(f"y validation Preprocessed Dimensions: {y_val_prep.shape}")

print(f"X test Preprocessed Dimensions: {X_test_prep.shape}")

### Training the model using the best hyperparameters found so far by running descete experiments and Optuna study on the entire dataset

In [ ]:
train_dataset = TweetDataset(X_train_prep, y_train_prep, tokenizer, best_max_length)
val_dataset = TweetDataset(X_val_prep, y_val_prep, tokenizer, best_max_length)

train_loader = DataLoader(train_dataset, batch_size=final_batch_size, sampler=get_random_sampler(train_dataset, seed))
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

final_model = AutoModelForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)
final_model.to(device)

optimizer = best_optimizer(final_model.parameters(), lr=final_lr, weight_decay=final_weight_decay, eps=final_eps)
final_results, best_state = train_model(final_model, train_loader, val_loader, optimizer=optimizer, n_epochs=4, device=device)

final_model.load_state_dict(best_state);

### Plotting the final BERT model learning curve and best validation set (on epoch) scores

In [ ]:
plot_epochs_scores(
    final_results,
    f"BERT, Optuna Tuning, Optimizer: {best_opt_name}, lr: {final_lr:.2e}, wd: {final_weight_decay:.2e}\nEPS: {final_eps:.2e}, Batch Size: {final_batch_size}, Token Max Length: {best_max_length}"
)

plot_best_scores(
    final_results,
    f"BERT, Optuna Best Validation Set Scores, Optimizer: {best_opt_name}, lr: {final_lr:.2e}\nWD: {final_weight_decay:.2e}, EPS: {final_eps:.2e}, Batch Size: {final_batch_size}, Token Max Length: {best_max_length}, Best Epoch: {final_results['Best Epoch Number']}"
)

### Predicting validation set's labels to calculate ROC curve and Confusion Matrix

In [ ]:
validation_set_tokens = tokenizer(list(X_val_prep), truncation=True, padding=True, max_length=best_max_length, return_tensors='pt')
y_val_preds, y_val_probs = evaluate(final_model, validation_set_tokens, device)

In [ ]:
y_val_tensor = torch.tensor(y_val_prep.values).float().unsqueeze(1)
y_val_true = y_val_tensor.cpu().numpy().flatten()

### Calculating and plotting ROC curve and Confusion Matrix for validation set

In [ ]:
plot_roc_curve(*calc_roc_curve(y_val_true, y_val_probs), "Validation Set")
plot_confusion_matrix(calc_confusion_matrix(y_val_true, y_val_preds))

### Predicting labels for test set using the final optimized BERT model

In [ ]:
test_set_bert = tokenizer(list(X_test_prep), truncation=True, padding=True, max_length=best_max_length, return_tensors='pt')
y_test_preds, _ = evaluate(final_model, test_set_bert, device)

### Storing the label predictions among with the tweet IDs inside the `submission.csv` file

In [ ]:
pd.DataFrame({
    'ID': df_test['ID'],
    'Label': y_test_preds,
}).to_csv('submission.csv', index=False)